---
title: Lattice Vibrations Assignment
authors: gvarnavides
date: 2026-06-02
---

In this assignment, we will investigate the vibrational motion of atoms in a crystalline solid and connect these vibrations to quantum statistical physics.

We will model a crystal as a two-dimensional square lattice of atoms connected by harmonic springs.
Although this is a highly simplified model of a real solid, it already captures many important physical ideas:
- collective vibrational motion,
- normal modes,
- phonons,
- dispersion relations,
- and Bose–Einstein statistics.

The assignment also extends the molecular dynamics methods from the first assignment, where you used Velocity Verlet integration to simulate one-dimensional motion.

In [ ]:
import tqdm
import numpy as np
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
from scipy.interpolate import griddata
from lattice_vibrations import make_initial_displacement, velocity_verlet_2d, plot_lattice_snapshot, visualize_orbit, spring_force_2d



### Square Lattice of Springs

We consider a finite two-dimensional square lattice with fixed boundaries.
Each atom is connected to its nearest neighbors by identical harmonic springs.

If an atom is displaced from its equilibrium position, the springs exert restoring forces that tend to pull it back toward equilibrium.
Because all atoms are coupled together, a displacement of one atom produces collective motion throughout the lattice.

For example, for `interior_atoms=15`, the equilibrium (unperturbed) lattice is:

In [ ]:
#| label: phonon_initial_lattice
interior_atoms = 15
num_disp = (interior_atoms+2)**2 * 2

zero_displacements = np.zeros(num_disp)
plot_lattice_snapshot(zero_displacements,interior_atoms=interior_atoms);


Each atom can move in both the horizontal and vertical directions.
The complete lattice configuration is therefore described by a vector of displacements

$$
\mathbf{u}
=
(u_1,v_1,u_2,v_2,\dots),
$$
where $(u_i,v_i)$ denotes the displacement of atom $i$ from its equilibrium position.

### Initial Sinusoidal Perturbations

To excite collective lattice vibrations, we initialize the system with a sinusoidal displacement pattern.

The displacement field is given by

$$
u(x,y)
=
A
\sin(k_x x)
\sin(k_y y),
$$

where

- $A$ is the displacement amplitude,
- $k_x=$ and $k_y$ are wavevectors,
- $n_x$ and $n_y$ are integer lattice coordinates.

These sinusoidal patterns correspond to standing-wave vibrational modes of the lattice.

For example, for $(n_x,n_y)=(3,1)$:

In [ ]:
#| label: phonon_perturbed_lattice
uv_flat, kx,ky = make_initial_displacement(
    nx=15,
    ny=1,
    interior_atoms=interior_atoms
)
plot_lattice_snapshot(uv_flat,interior_atoms=interior_atoms,scale=1);

Different choices of $(n_x,n_y)$ produce different vibrational wavelengths and therefore different oscillation frequencies.

### Velocity Verlet Integration

To evolve the lattice in time, we once again use Velocity Verlet integration.

This is the same numerical integration method used in the first assignment for one-dimensional particle dynamics.
The main difference is that we now evolve many coupled degrees of freedom simultaneously.

You can use the provided  `velocity_verlet_2d` [function](/lattice-vibrations-source-code) as follows:

In [ ]:
#| label: phonon_example_orbit
orbit_qs, orbit_ps, orbit_ts = velocity_verlet_2d(uv_flat,spring_force_2d)

visualize_orbit(
    orbit_qs,
    orbit_ts,
    every=len(orbit_qs)//11,
    scale=10
);

Because the forces are harmonic, the lattice executes periodic collective oscillations known as normal modes.

### Part 3a: 2D Dispersion Relation

In this exercise, we compute the phonon dispersion relation of the 2D square lattice by directly measuring the frequencies of its normal modes.

Each sinusoidal initial displacement corresponds to a mode with wave-vector
$$
\mathbf{k} = (k_x, k_y),
$$
set by the sub-harmonic indices $(n_x,n_y)$.

For each mode:

- Initialize the lattice with the sinusoidal displacement.
- Evolve it using `velocity_verlet_2d`.
- Extract the oscillation period $T$ and compute
    
  $$
  \omega = \frac{2\pi}{T}.
  $$

Compute and plot $\omega(\mathbf{k})$ for all integer values:

$$
n_x, n_y \in \{- (\text{interior\_atoms}+1), \dots, -1, 1, \dots, (\text{interior\_atoms}+1)\},
$$

excluding the zero mode $(n_x,n_y)=(0,0)$.
This produces a discrete 2D map of the vibrational spectrum.

Comment on how the frequency changes with wavelength and identify which modes correspond to long-wavelength (low-$\omega$) excitations.

__Hint__:
You do not need to compute all modes explicitly.  
The system is symmetric under sign changes of $(n_x,n_y)$, so frequencies satisfy
$$
\omega(k_x,k_y) = \omega(-k_x,k_y) = \omega(k_x,-k_y).
$$

In [ ]:
#Initial test
uv_flat, kx,ky = make_initial_displacement(nx=4,ny=1,interior_atoms=interior_atoms) # create initial grid
orbit_qs, orbit_ps, orbit_ts = velocity_verlet_2d(uv_flat,spring_force_2d) # evolve grid
T_orbit = orbit_ts[-1]
w_orbit = 2*np.pi/T_orbit
print(w_orbit)

In [ ]:
def Get_angfreq(nx, ny, interior_atoms):
    uv_flat, kx,ky = make_initial_displacement(nx=nx,ny=ny,interior_atoms=interior_atoms) # create initial grid
    orbit_qs, orbit_ps, orbit_ts = velocity_verlet_2d(uv_flat,spring_force_2d) # evolve grid
    if np.max(orbit_qs) < 1e-6: #check if there is an actual oscilation
        return np.nan
    else:
        return 2*np.pi/orbit_ts[-1] # return angular frequency


In [ ]:
n_max = interior_atoms+1
n = np.arange(1,n_max+1,1)
print(n)

In [ ]:
w = np.zeros([n_max,n_max]) # The frequencies are the same for all quadrants of the nx,ny plane

for i in range(len(n)):
    for j in range(len(n)):
        w[i,j] = Get_angfreq(n[i], n[j], interior_atoms)

print(w)


In [ ]:
w_tot = np.zeros([2*n_max+1,2*n_max+1])
np.shape(w_tot)
w_tot[17:33,17:33] = w
w_tot[0:16,17:33] = np.rot90(w)
w_tot[0:16,0:16] = np.rot90(np.rot90(w))
w_tot[17:33,0:16] = np.rot90(np.rot90(np.rot90(w)))
w_tot[16,:],w_tot[:,16] = np.nan,np.nan

In [ ]:
kx_max = np.pi*(n_max+0.5)/ (interior_atoms + 1) # Only for centering plotting

extent = [-kx_max, kx_max, -kx_max, kx_max]
im = plt.imshow(w_tot, extent=extent, cmap='viridis', origin='upper')
plt.colorbar(im)
ticks = np.linspace(-kx_max, kx_max, 4)
plt.xticks(ticks)
plt.yticks(ticks)
plt.xlabel("kx")
plt.ylabel("ky")
plt.savefig("frequency with k's.png")
plt.show()
print(np.max(w[14,14]))

### Part 3b: Bose Einstein Occupations

Using the dispersion relation $\omega(\mathbf{k})$, treat each vibrational mode as a quantum harmonic oscillator with [](wiki:Bose_Einstein_statistics)
$$
n(\omega,T) = \frac{1}{\exp(\hbar \omega / k_B T)-1}.
$$

1. For $T \in [0.2, 5.0]$, plot $n(\omega,T)$ as a function of $\omega$.
2. For a fixed temperature (e.g. $T=5.0$), plot $n(\mathbf{k})$.

Comment on why low-frequency modes dominate the occupation and how this changes with temperature.

deel 1:

In [ ]:
def ocupation(w,T): #Calculate ocupation levels
    a = np.e**(w/T)-1
    return 1/a
print(ocupation(1,1))

In [ ]:
# Initialize some variables
N = 100
T_array = np.linspace(0.2,5,6)
w_array = np.linspace(0.01,0.5,N)


for i in range(0,6): # plot for 6 different temperatures n as a function of w
    T = T_array[i]
    plt.plot(w_array,ocupation(w_array,T),label="T = %.1f"%T)

# some plotting code
plt.xlabel("w")
plt.ylabel("ocupation number")
plt.yscale("log")
plt.legend()
plt.savefig("n(w).png")

Deel 2:

In [ ]:
# plot ocupation for different wavevectors
T = 5
kx_max = np.pi*(n_max+0.5)/ (interior_atoms + 1) # Only for centering plotting

extent = [-kx_max, kx_max, -kx_max, kx_max]
plt.figure(figsize=(8, 8))
im = plt.imshow(ocupation(w_tot, T), extent=extent, cmap='viridis', origin='upper', norm=LogNorm()) #plot ocupation numbers based on wavevector on a logscale
plt.colorbar(im,label="ocupation") # plotting of colorbar and matrix

# some visualization code
ticks = np.linspace(-kx_max, kx_max, 4)
plt.xticks(ticks)
plt.yticks(ticks)
plt.xlabel("kx")
plt.ylabel("ky")
plt.show()
plt.savefig("ocupation versus wavevector.png")

In [ ]:
# seeing whether the function n(k) only depends on the length of k vector
n_2 = np.array([])
k_2 = np.array([])
for i in n-1: # Loop over all n's to calculate all frequencies and wavevector
    for j in n-1:
        n_2 = np.append(n_2,ocupation(w[i,j],T))
        k = np.sqrt((np.pi/ (interior_atoms + 1))**2*(i**2+j**2)) # calculate the length of k vector
        k_2 = np.append(k_2,k)


# print(w_2)
plt.plot(n_2,k_2,"c.")
plt.xlabel("w")
plt.ylabel("ocupation number")
plt.savefig("length kvector.png")
# The length of k vector is not the only determining factor in the ocupation number.

### Part 3c: Phonon Multiplicity

Compute the phonon multiplicity
$$
g(\omega) = \int \delta(\omega - \omega(\mathbf{k}))\, d\mathbf{k}
$$
using a histogram of your computed dispersion relation.

Plot $g(\omega)$ and identify any sharp features (Van_Hove singularities).

In [ ]:
def dirac_delta(a,b): # define a dirac delta function
    if np.abs(a-b) < 0.01:
        return 1
    else:
        return 0

In [ ]:
print(np.arange(0.01,0.5,0.05))

In [ ]:
w_2 = np.arange(0.01,3,0.03) # define test arrayl
g_w = np.zeros(len(w_2))

for l in range(len(w_2)):
    a=0
    for i in n - 1:
        for j in n - 1:
            a += dirac_delta(w_2[l],w[i,j]) # sum over all frequencies
    g_w[l] = a

# normalize g_w
g_w = g_w / (np.sum(g_w)*0.03)
print(np.sum(g_w)*0.03)
# plt.plot(w_2,g_w)
# Plotting the histogram
edges = np.append(w_2 - 0.015, w_2[-1] + 0.015)
plt.stairs(g_w, edges, fill=True, color='skyblue', alpha=0.6, baseline=0)
plt.stairs(g_w, edges, fill=False, color='black', linewidth=1.2, baseline=0)
plt.ylabel("g(w)")
plt.xlabel("w")
plt.savefig("histogram g_w.png")

### Part 3d: Quantum Statistics

Compute the internal energy of the lattice using Bose–Einstein statistics:
$$
\langle E \rangle =
\sum_{\mathbf{k}} \hbar \omega(\mathbf{k})
\left(
\frac{1}{\exp(\hbar \omega(\mathbf{k})/k_B T)-1}
+
\frac{1}{2}
\right).
$$

From this, compute and plot the heat capacity
$$
C_v(T) = \frac{d\langle E \rangle}{dT}.
$$

Comment on the approach to the classical Dulong-Petit limit at high temperature and the role of the discrete phonon spectrum at low temperature.